# Step 2 — LLM Annotation: Sentiment & Topic Analysis

This notebook explores the results of the LLM annotation pass run by `src/analysis/annotate.py`.
It assumes that:
1. `annotate.py` has been run and `data/processed/llm_annotations/*.jsonl` files exist.
2. `aggregate_features.py` has been run and `data/processed/features/nlp_features.parquet` exists.

**Sections:**
1. Load & inspect raw annotations
2. Sentiment time-series by source and speaker role
3. Topic heatmap across quarters
4. Representative key quotes per topic per quarter
5. Validation sample — manual label spot-check
6. NLP feature summary (what goes into Step 3)

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from src.utils.config import FEATURES_DIR, LLM_ANNOTATIONS_DIR

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

SOURCE_TYPES = ['transcripts', 'news', 'reports_40f', 'reports_quarterly']
TOPICS = [
    'NIM', 'credit_quality', 'capital', 'US_retail', 'Canadian_personal',
    'wealth_wholesale', 'regulatory_AML', 'macro_outlook', 'cost_efficiency',
    'guidance', 'M_and_A', 'other',
]
print('Setup complete')

## 1. Load raw annotations

In [ ]:
records = []
for src in SOURCE_TYPES:
    path = LLM_ANNOTATIONS_DIR / f'{src}.jsonl'
    if not path.exists():
        print(f'  {src}: NOT FOUND — run annotate.py first')
        continue
    n = 0
    with path.open() as fh:
        for line in fh:
            try:
                r = json.loads(line)
            except Exception:
                continue
            r['_src'] = src
            r['_sentiment'] = r.get('output', {}).get('sentiment', '')
            r['_score'] = r.get('output', {}).get('sentiment_score', None)
            r['_topics'] = r.get('output', {}).get('topics', [])
            r['_key_quote'] = r.get('output', {}).get('key_quote', '')
            r['_role'] = (r.get('speaker') or {}).get('role', '')
            r['_name'] = (r.get('speaker') or {}).get('name', '')
            records.append(r)
            n += 1
    print(f'  {src}: {n:,} annotations')

df = pd.DataFrame(records)
df['_score'] = pd.to_numeric(df['_score'], errors='coerce')
df['fiscal_quarter'] = df['fiscal_quarter'].fillna(df.get('td_fiscal_quarter_hint', ''))
print(f'\nTotal: {len(df):,} annotated chunks')
df.head(3)

In [ ]:
# Sentiment label distribution by source
pd.crosstab(df['_src'], df['_sentiment'], normalize='index').round(3).style.background_gradient(cmap='RdYlGn', axis=1)

## 2. Sentiment time-series

In [ ]:
# --- Quarterly mean sentiment by source ---
pivot = (
    df.groupby(['fiscal_quarter', '_src'])['_score']
    .mean()
    .unstack('_src')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], marker='o', markersize=4, label=col)
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_title('Quarterly Mean Sentiment Score by Source (−1 negative → +1 positive)')
ax.set_xlabel('Fiscal Quarter')
ax.set_ylabel('Mean sentiment score')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Transcript sentiment split: CEO prep vs CFO prep vs analyst Q&A ---
tr = df[df['_src'] == 'transcripts'].copy()

def role_section_label(row):
    role = row['_role']
    sec = row.get('section', '')
    if sec == 'prepared_remarks' and role == 'ceo':
        return 'CEO prepared'
    if sec == 'prepared_remarks' and role == 'cfo':
        return 'CFO prepared'
    if sec == 'qa' and role == 'analyst':
        return 'Analyst Q&A'
    if sec == 'qa' and role in ('ceo', 'cfo', 'other_exec'):
        return 'Exec Q&A'
    return None

tr['label'] = tr.apply(role_section_label, axis=1)
tr_filtered = tr[tr['label'].notna()]

pivot_tr = (
    tr_filtered.groupby(['fiscal_quarter', 'label'])['_score']
    .mean()
    .unstack('label')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
colors = {'CEO prepared': '#1f77b4', 'CFO prepared': '#ff7f0e',
          'Exec Q&A': '#2ca02c', 'Analyst Q&A': '#d62728'}
for col in pivot_tr.columns:
    ax.plot(pivot_tr.index, pivot_tr[col], marker='o', markersize=5,
            label=col, color=colors.get(col))
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_title('Transcript Sentiment by Speaker Role & Section')
ax.set_xlabel('Fiscal Quarter')
ax.set_ylabel('Mean sentiment score')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Topic heatmap

In [ ]:
# Explode topics (each chunk can have multiple)
df_topics = df.explode('_topics').rename(columns={'_topics': 'topic'})
df_topics = df_topics[df_topics['topic'].isin(TOPICS)]

topic_heatmap = (
    df_topics.groupby(['fiscal_quarter', 'topic'])
    .size()
    .unstack('topic', fill_value=0)
    .sort_index()
)
# Normalise each row to show share within quarter
topic_heatmap_pct = topic_heatmap.div(topic_heatmap.sum(axis=1), axis=0)

# Drop 'other' from the plot for clarity
cols_to_show = [c for c in topic_heatmap_pct.columns if c != 'other']

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(
    topic_heatmap_pct[cols_to_show].T,
    ax=ax,
    cmap='YlOrRd',
    fmt='.1%',
    annot=True,
    annot_kws={'size': 8},
    linewidths=0.4,
    cbar_kws={'label': 'Share of quarter chunks'},
)
ax.set_title('Topic Share Heatmap by Fiscal Quarter (all sources, excluding "other")')
ax.set_xlabel('Fiscal Quarter')
ax.set_ylabel('Topic')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Topic sentiment heatmap — is the tone around each topic positive or negative?
topic_sentiment = (
    df_topics.groupby(['fiscal_quarter', 'topic'])['_score']
    .mean()
    .unstack('topic')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(
    topic_sentiment[cols_to_show].T,
    ax=ax,
    cmap='RdYlGn',
    center=0,
    fmt='.2f',
    annot=True,
    annot_kws={'size': 8},
    linewidths=0.4,
    vmin=-1, vmax=1,
    cbar_kws={'label': 'Mean sentiment score'},
)
ax.set_title('Mean Sentiment Score per Topic per Quarter')
ax.set_xlabel('Fiscal Quarter')
ax.set_ylabel('Topic')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Representative key quotes per topic

In [ ]:
# For each topic, show the most negative and most positive key quote across all quarters
SHOW_TOPICS = ['NIM', 'credit_quality', 'regulatory_AML', 'macro_outlook', 'capital']

for topic in SHOW_TOPICS:
    sub = df_topics[df_topics['topic'] == topic].dropna(subset=['_score', '_key_quote'])
    if sub.empty:
        continue
    most_neg = sub.loc[sub['_score'].idxmin()]
    most_pos = sub.loc[sub['_score'].idxmax()]
    print(f'\n── {topic} ──')
    print(f'  Most positive [{most_pos["fiscal_quarter"]}] ({most_pos["_score"]:+.2f})')
    print(f'  "{most_pos["_key_quote"]}"')
    print(f'  Most negative [{most_neg["fiscal_quarter"]}] ({most_neg["_score"]:+.2f})')
    print(f'  "{most_neg["_key_quote"]}"')

## 5. Validation sample — manual spot-check

Sample 30 chunks (10 each from transcripts, news, reports_quarterly). Review the LLM label and decide if you agree.

In [ ]:
# Load the original chunk text alongside annotations for comparison
import json as _json
from src.utils.config import CHUNKS_DIR

chunk_lookup: dict[str, str] = {}
for src in SOURCE_TYPES:
    path = CHUNKS_DIR / f'{src}.jsonl'
    if not path.exists():
        continue
    with path.open() as fh:
        for line in fh:
            try:
                c = _json.loads(line)
                chunk_lookup[c['chunk_id']] = c.get('text', '')[:600]
            except Exception:
                pass

validation_sample = (
    df[df['_src'].isin(['transcripts', 'news', 'reports_quarterly'])]
    .groupby('_src', group_keys=False)
    .apply(lambda g: g.sample(min(10, len(g)), random_state=42))
    .reset_index(drop=True)
)
validation_sample['chunk_text_preview'] = validation_sample['chunk_id'].map(
    lambda cid: chunk_lookup.get(cid, '')[:300]
)

display_cols = ['_src', 'fiscal_quarter', '_role', '_sentiment', '_score', '_topics', '_key_quote', 'chunk_text_preview']
validation_sample[display_cols].style.set_properties(**{'text-align': 'left', 'font-size': '11px'})

## 6. NLP feature summary (Step 3 inputs)

In [ ]:
feat_path = FEATURES_DIR / 'nlp_features.parquet'
if not feat_path.exists():
    print('Run aggregate_features.py first: python -m src.analysis.aggregate_features')
else:
    feat = pd.read_parquet(feat_path)
    print(f'Feature matrix: {feat.shape[0]} quarters × {feat.shape[1]} columns')
    feat.head()

In [ ]:
if feat_path.exists():
    # Sentiment columns over time
    sent_cols = [c for c in feat.columns if 'sentiment_mean' in c]
    fig, ax = plt.subplots(figsize=(14, 5))
    for col in sent_cols:
        ax.plot(feat['fiscal_quarter'], feat[col], marker='o', markersize=4, label=col.replace('_sentiment_mean', ''))
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax.set_title('NLP Feature: Quarterly Mean Sentiment (Step 3 inputs)')
    ax.set_xlabel('Fiscal Quarter')
    ax.set_ylabel('Mean sentiment score')
    ax.legend(fontsize=9)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
if feat_path.exists():
    # Topic entropy — how concentrated is the narrative each quarter?
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(feat['fiscal_quarter'], feat['topic_entropy'], color='steelblue', alpha=0.8)
    ax.set_title('Topic Entropy per Quarter (high = broad discussion, low = concentrated)')
    ax.set_xlabel('Fiscal Quarter')
    ax.set_ylabel('Shannon entropy (bits)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
if feat_path.exists():
    # Full feature matrix — show missing values heatmap
    numeric_cols = feat.select_dtypes(include='number').columns
    missing_pct = feat[numeric_cols].isna().mean()
    print('Columns with missing values:')
    print(missing_pct[missing_pct > 0].sort_values(ascending=False).to_string())
    if (missing_pct > 0).sum() == 0:
        print('No missing values — feature matrix is complete.')